# Autoencoder Example 4 (PyTorch) — Variational Autoencoder (VAE)

PyTorch version of `variational_autoencoder.ipynb`. A VAE turns the autoencoder into a
**generative model**:

1. The encoder outputs a **distribution** ($\mu$, $\log\sigma^2$) per image; we sample
   $z = \mu + \sigma\,\epsilon$ (the reparameterization trick).
2. A **KL term** pulls each distribution toward $\mathcal{N}(0,I)$, so the latent space is
   tidy and we can sample $z\sim\mathcal{N}(0,I)$ to generate new digits.

Loss = reconstruction (per-pixel BCE, summed) + KL divergence. We use a **2D latent
space** so it can be visualised.

> Needs `torch` + `torchvision`. Trains in ~1 minute on CPU.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch', torch.__version__, '| device:', device)

In [ ]:
transform = transforms.ToTensor()
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
LATENT_DIM = 2

## Model

`encode` returns `mu` and `logvar`; `reparameterize` samples `z`; `decode` reconstructs.
Two linear heads (`fc_mu`, `fc_logvar`) sit on top of the shared encoder trunk.

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU())
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        self.dec = nn.Sequential(nn.Linear(latent_dim, 128), nn.ReLU(),
                                 nn.Linear(128, 256), nn.ReLU(),
                                 nn.Linear(256, 784), nn.Sigmoid())

    def encode(self, x):
        h = self.enc(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + std * torch.randn_like(std)

    def decode(self, z):
        return self.dec(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

model = VAE().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

## Train

Reconstruction = summed BCE over pixels; KL has the standard closed form
`-0.5 * sum(1 + logvar - mu^2 - exp(logvar))`.

In [ ]:
EPOCHS = 25
for epoch in range(1, EPOCHS + 1):
    model.train(); tot = 0.0
    for images, _ in train_loader:
        x = images.view(images.size(0), -1).to(device)
        x_hat, mu, logvar = model(x)
        recon = F.binary_cross_entropy(x_hat, x, reduction='sum') / x.size(0)
        kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
        loss = recon + kl
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        tot += loss.item() * x.size(0)
    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:2d}/{EPOCHS}  loss/sample={tot / len(train_dataset):.2f}')

## Latent space, coloured by digit

In [ ]:
model.eval()
xs = torch.stack([test_dataset[i][0] for i in range(5000)]).view(5000, -1).to(device)
ys = np.array([test_dataset[i][1] for i in range(5000)])
with torch.no_grad():
    mu, _ = model.encode(xs)
mu = mu.cpu().numpy()
plt.figure(figsize=(6, 5))
sc = plt.scatter(mu[:, 0], mu[:, 1], c=ys, cmap='tab10', s=4, alpha=0.6)
plt.colorbar(sc, label='digit'); plt.xlabel('z[0]'); plt.ylabel('z[1]')
plt.title('VAE latent space (test set)'); plt.show()

## Generate new digits across the latent grid

In [ ]:
g = 15
grid = np.linspace(-2.5, 2.5, g).astype('float32')
canvas = np.zeros((28 * g, 28 * g))
model.eval()
with torch.no_grad():
    for i, zi in enumerate(grid[::-1]):
        for j, zj in enumerate(grid):
            z = torch.tensor([[zj, zi]], device=device)
            img = model.decode(z).cpu().numpy().reshape(28, 28)
            canvas[i*28:(i+1)*28, j*28:(j+1)*28] = img
plt.figure(figsize=(8, 8))
plt.imshow(canvas, cmap='gray'); plt.axis('off')
plt.title('Digits decoded across the 2D latent grid'); plt.show()

### Things to try

- Increase `LATENT_DIM` (e.g. 16) for sharper digits.
- Scale the KL term (`beta * kl`) for a **β-VAE**.
- Use convolutional encoder/decoder.
- Compare random samples with the plain autoencoder's.